# Classify BedMethyl Files from NanoDx using CrossNN
*Script: J. Hench IfP 2025-2026*

* Tested on x64, meqneuropatlp31, Ubuntu 20.04
* activate the environment with the following commands
```
source /applications/miniconda/nanodx_crossnn_venv/bin/activate
eval "$(mamba shell hook --shell bash)"
mamba activate crossnn01
cd /applications/nanodx_versions/nanoDx-master-crossnn-20250728/nnstandalone/
jupyter-notebook
```
**20260203**
*However,* it does not seem to be possible to transfer a trained model (GPU-trained with torch) from computer A (trained on GPU 1) to computer B (should classify on GPU 0).

adapted code to read cg

In [1]:
# pathes
nnmodelpath='/applications/crossNN_versions/crossNN-master_20250911/models/ND_IfP_20250912_NN.pkl'
inputpath='/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_2026_02_04'
outputpath='/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_output_20260204' 
verbose=False

In [2]:
# make the pathes if they don't exist
import os
os.makedirs(outputpath, exist_ok=True)

In [3]:
def displayhead(mylist): # display top n list entries
    if (len(mylist)>3):
        display(mylist[:3])
    else:
        display(mylist)

In [4]:
import fnmatch
pattern = "*-methoverlap.tsv"
cnt=10
c=0
pathlist=[]
filenamelist=[]
for root, dirs, files in os.walk(inputpath):
    for filename in fnmatch.filter(files, pattern):
        pathlist.append(root+"/")
        filenamelist.append(filename)

In [5]:
displayhead(filenamelist[:3])

['blank_1_0p0_0d20-methoverlap.tsv',
 'blank_1_0p0_0d21-methoverlap.tsv',
 'blank_1_0p0_0d22-methoverlap.tsv']

In [6]:
displayhead(pathlist[:3])

['/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_2026_02_04/blank_1_0p0_0d20/run1/',
 '/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_2026_02_04/blank_1_0p0_0d21/run1/',
 '/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_2026_02_04/blank_1_0p0_0d22/run1/']

In [7]:
# dependencies
from NN_model import NN_classifier
import pandas as pd
from IPython.display import clear_output # substitute for jupyterlab, tqdm not working properly

In [8]:
# functions
def classify_methoverlaptsv(nnmodelpath,methoverlaptsv,inputpath,outputpath,verbose):
    basename=methoverlaptsv.replace('-methoverlap.tsv','')
    methoverlaptsvpath=inputpath+methoverlaptsv
    votescsv=outputpath+"/"+basename+'-votes.tsv'
    classificationtxt=outputpath+"/"+basename+'-classification.tsv'
    if verbose:
        print(votescsv)
        print(classificationtxt)

    NN = NN_classifier(nnmodelpath)
    
    methoverlapdf=pd.read_csv(methoverlaptsvpath,sep='\t',header=None)
    if verbose:
        display(methoverlapdf)
    methoverlapdf.columns = ['probe_id','methylation_call']
    if verbose:
        display(methoverlapdf)
    
    # remove last columns (contain strange data)
    # methoverlapdf = methoverlapdf.drop(index=[4992,4993,4994,4995,4996,4997,4998,4999])
    if verbose:
        display(methoverlapdf)

    # replace 0's in df with -1
    methoverlapdf = methoverlapdf.replace(0, -1)
    if verbose:
        display(methoverlapdf)
    
    #predictions, class_labels, n_features = NN.predict_from_bedMethyl(snakemake.input["bed"])
    #predictions, class_labels, n_features = NN.predict_from_bedMethyl(bedmethylpath)
    predictions, class_labels, n_features = NN.predict(methoverlapdf)
    
    
    # write predictions to table
    df = pd.DataFrame({'class': class_labels, 'score': predictions, 'num_features': n_features})
    
    
    #df.to_csv(snakemake.output['votes'], sep='\t')
    df.to_csv(votescsv, sep='\t')
    
    if verbose:
        display(df)
    
    # write summary to txt file
    summary = ['Number of features: ' + str(n_features),
               'Predicted Class: ' + str(class_labels[0]),
               "Score: " + str(predictions[0])
              ]
    
    
    #with open(snakemake.output["txt"], 'w') as f:
    with open(classificationtxt, 'w') as f:
      f.write("\n".join(summary))

In [9]:
# caselist=[
#     'B2023_41230_0_4_20250806270k-methoverlap.tsv',
#     'B2023_41230_10_20250806270k-methoverlap.tsv',
#     'B2023_41230_2_20250806270k-methoverlap.tsv',
#     'B2023_41230_50_20250806270k-methoverlap.tsv',
#     'blank_DNeasy_01_0_20250806270k-methoverlap.tsv',
#     'blank_Maxwell_0_20250806270k-methoverlap.tsv',
#     'K2024_1964_50_20250806270k-methoverlap.tsv',
#     'M2024_118_50_20250806270k-methoverlap.tsv',
#     'M2024_1250_50_20250806270k-methoverlap.tsv',
#     'M2024_1573_50_20250806270k-methoverlap.tsv',
#     'M2024_2251_50_20250806270k-methoverlap.tsv',
#     'M2024_3714_0_4_20250806270k-methoverlap.tsv',
#     'M2024_3714_10_20250806270k-methoverlap.tsv',
#     'M2024_3714_2_20250806270k-methoverlap.tsv',
#     'M2024_3714_50_20250806270k-methoverlap.tsv',
#     'M2024_3878_50_20250806270k-methoverlap.tsv',
#     'M2024_4137_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4137_100_20250806270k-methoverlap.tsv',
#     'M2024_4137_10_20250806270k-methoverlap.tsv',
#     'M2024_4137_2_20250806270k-methoverlap.tsv',
#     'M2024_4137_500_20250806270k-methoverlap.tsv',
#     'M2024_4137_50_20250806270k-methoverlap.tsv',
#     'M2024_4137_75_20250806270k-methoverlap.tsv',
#     'M2024_4244_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4244_10_20250806270k-methoverlap.tsv',
#     'M2024_4244_2_20250806270k-methoverlap.tsv',
#     'M2024_4244_50_20250806270k-methoverlap.tsv',
#     'M2024_4246_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4246_100_20250806270k-methoverlap.tsv',
#     'M2024_4246_150_20250806270k-methoverlap.tsv',
#     'M2024_4246_2_20250806270k-methoverlap.tsv',
#     'M2024_4246_4_20250806270k-methoverlap.tsv',
#     'M2024_4246_50_20250806270k-methoverlap.tsv',
#     'M2024_4435_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4435_100_20250806270k-methoverlap.tsv',
#     'M2024_4435_10_20250806270k-methoverlap.tsv',
#     'M2024_4435_2_20250806270k-methoverlap.tsv',
#     'M2024_4435_500_20250806270k-methoverlap.tsv',
#     'M2024_4435_50_20250806270k-methoverlap.tsv',
#     'M2024_4435_75_20250806270k-methoverlap.tsv',
#     'M2024_4547_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4547_150_20250806270k-methoverlap.tsv',
#     'M2024_4547_2_20250806270k-methoverlap.tsv',
#     'M2024_4547_50_20250806270k-methoverlap.tsv',
#     'M2024_4547_6_20250806270k-methoverlap.tsv',
#     'M2024_4656_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4656_100_20250806270k-methoverlap.tsv',
#     'M2024_4656_10_20250806270k-methoverlap.tsv',
#     'M2024_4656_2_20250806270k-methoverlap.tsv',
#     'M2024_4656_500_20250806270k-methoverlap.tsv',
#     'M2024_4656_50_20250806270k-methoverlap.tsv',
#     'M2024_4656_75_20250806270k-methoverlap.tsv',
#     'M2024_4657_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4657_100_20250806270k-methoverlap.tsv',
#     'M2024_4657_10_20250806270k-methoverlap.tsv',
#     'M2024_4657_2_20250806270k-methoverlap.tsv',
#     'M2024_4657_500_20250806270k-methoverlap.tsv',
#     'M2024_4657_50_20250806270k-methoverlap.tsv',
#     'M2024_4657_75_20250806270k-methoverlap.tsv',
#     'M2024_4862_0_4_20250806270k-methoverlap.tsv',
#     'M2024_4862_10_20250806270k-methoverlap.tsv',
#     'M2024_4862_150_20250806270k-methoverlap.tsv',
#     'M2024_4862_2_20250806270k-methoverlap.tsv',
#     'M2024_4862_250_20250806270k-methoverlap.tsv',
#     'M2024_4862_50_20250806270k-methoverlap.tsv',
#     'M2024_623_50_20250806270k-methoverlap.tsv',
#     'M2024_738_50_20250806270k-methoverlap.tsv',
#     'M2024_837_50_20250806270k-methoverlap.tsv',
#     'M2025_1405_0_4_20250806270k-methoverlap.tsv',
#     'M2025_1405_10_20250806270k-methoverlap.tsv',
#     'M2025_1405_2_20250806270k-methoverlap.tsv',
#     'M2025_1405_50_20250806270k-methoverlap.tsv',
#     'M2025_1926_50_20250806270k-methoverlap.tsv',
#     'M2025_1934_50_20250806270k-methoverlap.tsv',
#     'M2025_28_0_4_20250806270k-methoverlap.tsv',
#     'M2025_28_10_20250806270k-methoverlap.tsv',
#     'M2025_28_2_20250806270k-methoverlap.tsv',
#     'M2025_28_50_20250806270k-methoverlap.tsv'
# ]

In [10]:
# caselist=[
#     'Sample_26_50p0_0d39',
#     'Sample_27_50p0_0d39'
# ]

In [11]:
# run the classifier
l=len(filenamelist)
for c in range(l):
    clear_output(wait=True)
    print("Processing "+filenamelist[c]+ " with NN ("+str(c+1)+"/"+str(l)+")")
    classify_methoverlaptsv(nnmodelpath,filenamelist[c],pathlist[c],outputpath,verbose)

Processing Sample_41_50p0_0d60-methoverlap.tsv with NN (3198/3198)


In [14]:
# todo
# Resultate tabellarisch umformatieren
headerline="sample ID\tDNA concentration\tbeta threshold\tNumber of features\tPredicted Class\tScore"
tablefile=outputpath+"/results_summary.tsv"
outfile=open(tablefile, 'w')
outfile.write(headerline+'\n')

for c in filenamelist:
    basename=c.replace('-methoverlap.tsv','')
    classificationtxt=outputpath+"/"+basename+'-classification.tsv'
    print(classificationtxt)
    try:
        with open(classificationtxt, 'r') as infile:
            content = infile.read()
        #print(content)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")
    # convert string into tab-separated line
    tableline=content.replace("Number of features: ","").replace("Predicted Class: ","\t").replace("Score: ","\t").replace("\n","")

    stringparts=basename.split("_")
    sampleid=stringparts[0]+"."+stringparts[1]
    print(len(stringparts))
    if len(stringparts)==4:
        dnaconc=stringparts[2]
        procdate=stringparts[3]
    elif len(stringparts)==5:
        dnaconc=stringparts[2]+"."+stringparts[3]
        procdate=stringparts[4]
    else:
        dnaconc=""
        procdate=""

    tableline=sampleid+"\t"+dnaconc+"\t"+procdate+"\t"+tableline
    
    print(tableline)
    outfile.write(tableline+'\n')
outfile.close()

/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_output_20260204/blank_1_0p0_0d20-classification.tsv
4
blank.1	0p0	0d20	4996	ATRT_SHH	0.09696975350379944
/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_output_20260204/blank_1_0p0_0d21-classification.tsv
4
blank.1	0p0	0d21	4996	ATRT_SHH	0.09696975350379944
/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_output_20260204/blank_1_0p0_0d22-classification.tsv
4
blank.1	0p0	0d22	4996	ATRT_SHH	0.09696975350379944
/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_output_20260204/blank_1_0p0_0d23-classification.tsv
4
blank.1	0p0	0d23	4996	ATRT_SHH	0.09696975350379944
/run/user/1000/gvfs/sftp:host=meqneuropatlp43.lan,user=minknow/RN22TBraid01/nanopore22TB01/MSA_evaluation_output_20260204/blank_1_0p0_0d24-classification.ts

In [ ]:
# todo
# bin files lesen + index, daraus dataframe binärisiert einstellbar.